# Naive Bayes phân loại nấm ăn được và nấm độc

Bộ dữ liệu Mushroom có các thuộc tính phân loại và nhãn `class` (`e`: ăn được, `p`: độc). Dùng Categorical Naive Bayes với mã hóa hạng mục được fit trên tập huấn luyện.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, classification_report, ConfusionMatrixDisplay)

DATA_PATH = Path("../data/mushrooms.csv")
df = pd.read_csv(DATA_PATH)
print(f"Kích thước dữ liệu: {df.shape}")
display(df.head())
print("\nKiểu dữ liệu và giá trị thiếu:")
display(pd.DataFrame({"kiểu dữ liệu": df.dtypes.astype(str), "giá trị thiếu": df.isna().sum()}))


## Khám phá dữ liệu và chia tập

In [ ]:
TARGET = "class"
print("Phân bố nhãn:")
display(df[TARGET].value_counts(dropna=False).rename(index={"e": "ăn được (e)", "p": "độc (p)"}).to_frame("số lượng"))
df_model = df.dropna(subset=[TARGET]).copy()
X = df_model.drop(columns=[TARGET]).astype("string")
y = df_model[TARGET].astype(str)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("Train:", X_train.shape, "Test:", X_test.shape)


## Mã hóa thuộc tính và huấn luyện Categorical Naive Bayes

OrdinalEncoder ánh xạ từng thuộc tính phân loại sang mã số hạng mục, sau đó CategoricalNB ước lượng xác suất mỗi giá trị theo lớp.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.naive_bayes import CategoricalNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
categorical_features = X.columns.tolist()
preprocessor = ColumnTransformer([("categorical", Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ordinal", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
]), categorical_features)])
model = Pipeline([("preprocess", preprocessor), ("classifier", CategoricalNB())])
model.fit(X_train, y_train)


## Đánh giá mô hình

Dữ liệu kiểm tra được giữ riêng khỏi quá trình huấn luyện. Báo cáo precision, recall, F1 và ma trận nhầm lẫn cho từng lớp.

In [ ]:
y_pred = model.predict(X_test)
print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification report:")
print(classification_report(y_test, y_pred, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues", values_format="d")
plt.title("Ma trận nhầm lẫn - Categorical Naive Bayes")
plt.tight_layout()
plt.show()


## Nhận xét

Dựa trên classification report và ma trận nhầm lẫn ở trên, hãy nhận xét mô hình phân loại **class (e/p)** tốt đến đâu, đặc biệt chú ý lớp ít xuất hiện và các trường hợp dự đoán sai.